In [ ]:
import scanpy as sc
vit_path = '/data2/project/bin_jip/Biomarker/data/vitiligo/vitiligo_data_all.h5ad'


In [3]:
vit = sc.read_h5ad(vit_path)

In [4]:
print(vit)

AnnData object with n_obs × n_vars = 60259 × 25074
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'BEST', 'Patient', 'Disease', 'Subtype', 'atlasID', 'ID', 'Group', 'Group2', 'Prof', 'Age', 'Sex', 'doublet_scores', 'doublet_class', 'percent.mt', 'unintegrated_clusters', 'seurat_clusters', 'harmony_clusters', 'annotation', 'CellID', 'new_anno'
    var: 'features'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'X_umap.harmony', 'X_umap.unintegrated'
    varm: 'PCs'


In [5]:
# 각 obs column에 대해 value counts를 출력 단, 30개 초과인 경우는 제외
for col in vit.obs.columns:
    if vit.obs[col].nunique() <= 50:
        print(f"Value counts for {col}:")
        print(vit.obs[col].value_counts())
        print()

Value counts for orig.ident:
orig.ident
Vit3       13992
Vit6        6451
Vit7810     6169
Vit4        5990
VIT_LS2     5950
VIT_LS1     5471
VIT_NL1     5307
VIT_NL2     5238
Vit5        2086
Vit1        1832
Vit2        1773
Name: count, dtype: int64

Value counts for BEST:
BEST
NA     32124
SNG    28135
Name: count, dtype: int64

Value counts for Patient:
Patient
NA       32124
Vit10     5905
Vit7      5049
VIT2      4501
Vit8      3915
VIT3      2500
VIT4      1643
Vit9      1542
VIT5      1096
VIT1      1038
VIT6       946
Name: count, dtype: int64

Value counts for Disease:
Disease
Vitiligo    60259
Name: count, dtype: int64

Value counts for Subtype:
Subtype
Vulgaris       31222
Universalis    29037
Name: count, dtype: int64

Value counts for atlasID:
atlasID
VIT3B     13992
VIT6B      6451
VIT4B      5990
VIT7B      3698
VIT2A      3111
VIT10C     2754
VIT5B      2086
VIT10A     2067
VIT3C      1922
VIT1B      1832
VIT2B      1773
VIT8C      1743
VIT2C      1390
VIT8B      1387

In [10]:
cell_mask = vit.obs["Group2"] != "NL"
vit_subset = vit[cell_mask].copy()

vit_subset.raw = None  # raw 제거
vit_subset.write_h5ad("/data2/project/bin_jip/Biomarker/data/vitiligo/vitiligo_data.h5ad")

In [11]:
vit_subset_path = "/data2/project/bin_jip/Biomarker/data/vitiligo/vitiligo_data.h5ad"
vit_subset = sc.read_h5ad(vit_subset_path)
print(vit_subset)

AnnData object with n_obs × n_vars = 49714 × 25074
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'BEST', 'Patient', 'Disease', 'Subtype', 'atlasID', 'ID', 'Group', 'Group2', 'Prof', 'Age', 'Sex', 'doublet_scores', 'doublet_class', 'percent.mt', 'unintegrated_clusters', 'seurat_clusters', 'harmony_clusters', 'annotation', 'CellID', 'new_anno'
    var: 'features'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'X_umap.harmony', 'X_umap.unintegrated'
    varm: 'PCs'


In [15]:
# 각 obs column에 대해 value counts를 출력 단, 30개 초과인 경우는 제외
for col in vit_subset.obs.columns:
    if vit_subset.obs[col].nunique() <= 50:
        print(f"Value counts for {col}:")
        print(vit_subset.obs[col].value_counts())
        print()

Value counts for orig.ident:
orig.ident
Vit3       13992
Vit6        6451
Vit7810     6169
Vit4        5990
VIT_LS2     5950
VIT_LS1     5471
Vit5        2086
Vit1        1832
Vit2        1773
Name: count, dtype: int64

Value counts for BEST:
BEST
NA     32124
SNG    17590
Name: count, dtype: int64

Value counts for Disease:
Disease
Vitiligo    49714
Name: count, dtype: int64

Value counts for Subtype:
Subtype
Vulgaris       28578
Universalis    21136
Name: count, dtype: int64

Value counts for atlasID:
atlasID
VIT3B     13992
VIT6B      6451
VIT4B      5990
VIT7B      3698
VIT10C     2754
VIT5B      2086
VIT3C      1922
VIT1B      1832
VIT2B      1773
VIT8C      1743
VIT2C      1390
VIT8B      1387
VIT10B     1084
VIT4C       782
VIT1C       701
VIT9C       682
VIT5C       676
VIT7C       500
VIT6C       271
Name: count, dtype: int64

Value counts for ID:
ID
VIT3B     13992
VIT6B      6451
VIT4B      5990
VIT7B      3698
VIT10C     2754
VIT5B      2086
VIT3C      1922
VIT1B      1832


In [18]:
# 'ID' column은 다 VIT{number}{A/B/C} 형태로 되어있음. 예시: VIT1A, VIT1B, VIT1C, VIT2A, VIT2B, VIT2C, ... 따라서 'ID' column에서 숫자 부분만 추출하여 'Patient' column으로 추가. 기존 Patient column은 제거 그리고 저장

vit_subset.obs['Patient'] = vit_subset.obs['ID'].str.extract(r'VIT(\d+)')[0]
vit_subset.write_h5ad("/data2/project/bin_jip/Biomarker/data/vitiligo/vitiligo_data.h5ad")


In [20]:
vit_subset.obs["Patient"].value_counts()


Patient
3     15914
4      6772
6      6722
7      4198
10     3838
2      3163
8      3130
5      2762
1      2533
9       682
Name: count, dtype: int64

In [23]:
# Subtype Vulgaris와 Universalis를 각각 0,1로 매핑하여 'label' column으로 추가.그리고 저장

vit_subset.obs['label'] = vit_subset.obs['Subtype'].map({'Vulgaris': '0', 'Universalis': '1'})
vit_subset.write_h5ad("/data2/project/bin_jip/Biomarker/data/vitiligo/vitiligo_data.h5ad")

In [24]:
vit_subset.obs["label"].value_counts()


label
0    28578
1    21136
Name: count, dtype: int64

In [28]:
# 각 환자별 new_anno분포 확인, 그리고 label과 new_anno의 관계도 확인

for label in vit_subset.obs['label'].unique():
    print(f"Label {label} new_anno value counts:")
    print(vit_subset.obs[vit_subset.obs['label'] == label]['new_anno'].value_counts())
    print()

Label 1 new_anno value counts:
new_anno
KC_spinous          4967
KC_basal            4167
Sweat_gland_cell    2635
FB                  1879
VEC                 1667
KC_granular         1513
KC_channel           785
KC_prolif            702
KC_mitotic           694
Pericyte             426
CD4_TRM              309
KC_follicular        282
KC_HLADR             262
LEC                  200
cDC2                 151
CD8_TRM              145
Macrophage            98
Melanocyte            63
NK_gdT                54
Langerhans            52
Treg                  27
cDC1                  23
Mast                  21
NKT                   14
Name: count, dtype: int64

Label 0 new_anno value counts:
new_anno
KC_basal            8351
KC_spinous          8222
KC_granular         2871
KC_mitotic          1467
KC_prolif           1296
FB                  1207
VEC                 1187
KC_channel          1111
Sweat_gland_cell    1006
Pericyte             482
KC_HLADR             383
LEC               